In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import KNNImputer

In [2]:
df = pd.read_csv('train.csv')
df.columns = df.columns.str.lower()
df.info()
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerid  891 non-null    int64  
 1   survived     891 non-null    int64  
 2   pclass       891 non-null    int64  
 3   name         891 non-null    str    
 4   sex          891 non-null    str    
 5   age          714 non-null    float64
 6   sibsp        891 non-null    int64  
 7   parch        891 non-null    int64  
 8   ticket       891 non-null    str    
 9   fare         891 non-null    float64
 10  cabin        204 non-null    str    
 11  embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


Index(['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp',
       'parch', 'ticket', 'fare', 'cabin', 'embarked'],
      dtype='str')

In [3]:
# Missing values (age / port)

# age (random forest)
df_copy = df
df_copy.info()
df_copy.head()

# 选取特征(pclass/sex/family/fare/embarked/title) 
# Q1：fare需要根据填充的age计算，但是age的缺失值填充又需要fare
# Solution：暂用avg_fare
# sex
df_copy['sex_numeric'] = df_copy.sex.apply(lambda x:1 if x == 'female' else 0)
# family
df_copy['family'] = df_copy.sibsp + df_copy.parch
# fare
df_copy['avg_fare'] = np.log1p(df_copy.fare / (df_copy.family + 1))
# port
df_copy['port'] = df_copy.embarked.fillna('C')
df_copy['port_numeric'] = df_copy.port.apply(lambda x:1 if x == 'C' else 2 if x == 'S' else 3)
# title
df_copy['title'] = df_copy.name.apply(lambda x:x.split(',')[1].split('.')[0].strip())
print(df_copy.title.value_counts())
def get_title(title):
    if title == 'Master':
        return 1
    elif title == 'Miss':
        return 2
    elif title == 'Mr':
        return 3
    elif title == 'Mrs':
        return 4
    else:
        return 5
df_copy['new_title'] = df_copy.title.apply(get_title)

# build model
model = RandomForestRegressor(max_depth=4,min_samples_leaf=3,random_state=3,oob_score=True)

# train model
train_df = df_copy[df_copy.age.notnull()]
X = train_df[['pclass','sex_numeric','family','avg_fare','port_numeric','new_title']]
y = train_df.age
model.fit(X,y)

# evaluate model
print(model.oob_score_)
y_train = model.predict(X)
print(r2_score(y,y_train))
print(model.feature_importances_)

# predict
test_df = df_copy[df_copy.age.isnull()]
X_test = test_df[['pclass','sex_numeric','family','avg_fare','port_numeric','new_title']]
model.predict(X_test)

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerid  891 non-null    int64  
 1   survived     891 non-null    int64  
 2   pclass       891 non-null    int64  
 3   name         891 non-null    str    
 4   sex          891 non-null    str    
 5   age          714 non-null    float64
 6   sibsp        891 non-null    int64  
 7   parch        891 non-null    int64  
 8   ticket       891 non-null    str    
 9   fare         891 non-null    float64
 10  cabin        204 non-null    str    
 11  embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady         

array([33.96553121, 32.45390841, 31.53650536, 29.05616239, 23.27910616,
       29.1629662 , 32.86588824, 23.16705079, 28.99898826, 28.88797243,
       29.1973834 , 34.62803149, 23.16705079, 28.95219192, 44.51768234,
       44.74811331,  4.83125675, 29.1629662 , 29.1973834 , 23.23548461,
       29.1973834 , 29.1973834 , 29.1629662 , 29.27417217, 17.97975187,
       29.1973834 , 34.62803149,  9.05198236, 30.75456637, 29.31087046,
       28.77101238,  5.60155603, 41.74449388, 44.47164832,  5.18515476,
        8.25812044, 31.04246096, 40.58099704, 36.28212594, 34.62803149,
       23.16705079, 29.97525669, 30.3068326 , 29.1629662 ,  7.84159622,
       21.95597694, 13.59499804, 16.3203399 , 29.31087046, 35.46697023,
       34.62803149, 23.16705079, 44.76429611, 23.16705079, 30.63549369,
       44.47164832, 44.74811331, 44.67721103, 23.16705079, 34.62803149,
       28.01305763, 29.1973834 , 32.742679  , 29.97525669,  9.29183892,
       35.39233188, 29.1629662 , 31.7427141 , 44.86490456, 29.05

In [3]:
# Custom Transformers
# 1.sex: binary 
# 2.port：missing value
# 3.age: missing value(random forest) + group
# 4.family: add columns + group
# 5.fare: allcation + log1p

# port
df['embarked'] = df.embarked.fillna('C') #这么处理对吗

# sex
class sextransformer(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None):
        return self
    def transform(self,X,y=None):
        X_copy = X.copy()
        X_copy['sex_numeric'] = X_copy['sex'].apply(lambda x:1 if x == 'female' else 0)
        return X_copy

# family
class familysizetransformer(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None):
        return self
    def transform(self,X,y=None):
        X_copy = X.copy()
        X_copy['family'] = X_copy.sibsp + X_copy.parch
        X_copy['family_size'] = X_copy.family.apply(lambda x:'alone' if x == 0 else 'small' if x <= 2 else 'large') 
        return X_copy

# age (title + fit + predict + fill in + group) （after family）
# 1. randomforest
class agetransformer(BaseEstimator,TransformerMixin):
    def __init__(self, max_depth=4, min_samples_leaf=3, random_state=3):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state
        self.model = RandomForestRegressor(max_depth=max_depth, min_samples_leaf=min_samples_leaf, random_state=random_state)
    def feature_process(self,X):
        X_copy = X.copy()
        X_copy['avg_fare'] = np.log1p(X_copy.fare / (X_copy.family + 1))
        X_copy['title'] = X_copy.name.apply(lambda x:x.split(',')[1].split('.')[0].strip())
        X_copy['embarked_numeric'] = X_copy.embarked.apply(lambda x:1 if x == 'C' else (2 if x == 'S' else 3))
        def get_title(title):
            if title == 'Master':
                return 1
            elif title == 'Miss':
                return 2
            elif title == 'Mr':
                return 3
            elif title == 'Mrs':
                return 4
            else:
                return 5
        X_copy['title_numeric'] = X_copy.title.apply(get_title)
        return X_copy
    def fit(self,X,y=None):
        X_processed = self.feature_process(X)
        train_df = X_processed[X_processed.age.notnull()]
        X_train = train_df[['pclass','sex_numeric','family','avg_fare','embarked_numeric','title_numeric']]
        y_train = train_df.age
        self.model.fit(X_train,y_train)
        return self
    def transform(self,X,y=None):
        X_processed = self.feature_process(X)
        test_df = X_processed[X_processed.age.isnull()]
        X_test = test_df[['pclass','sex_numeric','family','avg_fare','embarked_numeric','title_numeric']]
        predicted_age = self.model.predict(X_test) 
        X_processed['age_filled'] = X_processed.age
        age_bool = X_processed.age.isnull()
        X_processed.loc[age_bool,'age_filled'] = predicted_age 
        def get_age_group(age):
            if age <= 10:
                return 'Children'
            elif age <= 18:
                return 'Teenager'
            elif age <= 60:
                return 'Adult'
            else:
                return 'Senior'
        X_processed['age_group'] = X_processed.age_filled.apply(get_age_group)
        return X_processed
# 2. kkn
class agetransformer_kkn(BaseEstimator,TransformerMixin):
    def __init__(self, n_neighbors=5, weights='uniform'):
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.model = KNNImputer(n_neighbors=n_neighbors, weights=weights)
        self.feature = ColumnTransformer(
            [
                ('numeric',StandardScaler(),['avg_fare','family','age']),
                ('categories',OneHotEncoder(drop='first'),['sex_numeric','embarked_numeric','pclass','title_numeric']) # 如果test集出现没有出现的分类特征怎么处理
            ]
        )
    def feature_process(self,X):
        X_copy = X.copy()
        #fare
        X_copy['avg_fare'] = np.log1p(X_copy.fare / (X_copy.family + 1))
        #port
        X_copy['embarked_numeric'] = X_copy.embarked.apply(lambda x:1 if x == 'C' else (2 if x == 'S' else 3))
        #title
        X_copy['title'] = X_copy.name.apply(lambda x:x.split(',')[1].split('.')[0].strip())
        def get_title(title):
            if title == 'Master':
                return 1
            elif title == 'Miss':
                return 2
            elif title == 'Mr':
                return 3
            elif title == 'Mrs':
                return 4
            else:
                return 5
        X_copy['title_numeric'] = X_copy.title.apply(get_title)
        return X_copy
    def fit(self,X,y=None):
        X_features = self.feature_process(X)
        X_processed = self.feature.fit_transform(X_features) # 加上transform才能训练knn
        # 提取age的列索引
        self.age_idx_ = self.feature.transformers_[0][2].index('age')
        # 提取standardscaler参数——还原age
        age_scaler = self.feature.named_transformers_['numeric'] # 取出standardscaler转换器
        self.age_mean_ = age_scaler.mean_[self.age_idx_]
        self.age_std_ = age_scaler.scale_[self.age_idx_]
        # knn
        self.model.fit(X_processed)
        return self
    def transform(self,X,y=None):
        X_copy = X.copy()
        X_features = self.feature_process(X)
        X_processed = self.feature.transform(X_features) # 不能用fit，会用测试集的mean和std进行standardscaler
        filled_age = self.model.transform(X_processed) # 返回和输入维度一致的numpy数组，是scaler后的age
        age_scaled = filled_age[:,self.age_idx_]
        age_filled = age_scaled * self.age_std_ + self.age_mean_
        X_copy['age_filled'] = age_filled
        def get_age_group(age):
            if age <= 10:
                return 'Children'
            elif age <= 18:
                return 'Teenager'
            elif age <= 60:
                return 'Adult'
            else:
                return 'Senior'
        X_copy['age_group'] = X_copy.age_filled.apply(get_age_group)
        return X_copy

# fare (after age)  
# Q1.if a family is split across the training and cv sets
# Q2.if ticket numbers appear in the cv set that are not present in the training set
# Solution:
# during the fit process, calculate the fare for children and adults based on the ticket numbers in the training set
# during transform, fill cv set by matching ticket numbers with the training set
# if a ticket number in cv set does not exist in the training set, use the median fare within different class in training set.

class faretransformer(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None): 
        df_fare = X.groupby('ticket').agg(
            children_count = ('age_filled',lambda x:(x <= 10).sum()),   
            adult_count = ('age_filled',lambda x:(x > 10).sum()),
            family_fare = ('fare',max)
        )
        df_fare['adult_fare'] = 2 * df_fare.family_fare / (df_fare.children_count + 2 * df_fare.adult_count)
        df_fare['children_fare'] = df_fare.family_fare / (df_fare.children_count + 2 * df_fare.adult_count)
        # create a mapping between ticket and fare
        self.dic_adult_fare_ = df_fare.adult_fare.to_dict()    
        self.dic_children_fare_ = df_fare.children_fare.to_dict()
        # create mapping between pclass and adult/children median fare
        X['fare_allocated'] = X.apply(lambda x:self.dic_children_fare_[x.ticket] if x.age_filled <=10 else self.dic_adult_fare_[x.ticket], axis = 1)
        self.dic_adult_median_ = X[X.age_filled > 10].groupby('pclass')['fare_allocated'].median().to_dict()
        self.dic_children_median_ = X[X.age_filled <= 10].groupby('pclass')['fare_allocated'].median().to_dict() #如果训练集没有抽到class1会报错
        # calculate overall fare median as the final filling value
        self.fare_median_ = X.fare.median() 
        return self
    def transform(self,X,y=None):
        X_copy = X.copy()
        def log_fare(row):
            if row.age_filled <= 10:
                fare = self.dic_children_fare_.get(row.ticket, self.dic_children_median_.get(row.pclass, self.fare_median_))  
            else:
                fare = self.dic_adult_fare_.get(row.ticket,self.dic_adult_median_.get(row.pclass, self.fare_median_))
            return np.log1p(fare)
        X_copy['new_fare'] = X_copy.apply(log_fare, axis = 1)
        return X_copy

In [4]:
# Construct feature interactions (unfinished)
# 1.sex & pclass (before:0.76555/0.76794, after:0.78229/0.77751)
class sexpclasstransformer(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None):
        return self
    def transform(self,X,y=None):
        X_copy = X.copy()
        X_copy['sex_pclass'] = X_copy.sex.apply(lambda x:x.strip()) + X_copy.pclass.apply(lambda x:str(x).strip())
        return X_copy
# 2.sex & port (drop to 0.77511/0.77272)
class sexporttransformer(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None):
        return self
    def transform(self,X,y=None):
        X_copy = X.copy()
        X_copy['sex_port'] = X_copy.sex.apply(lambda x:x.strip()) + X_copy.embarked.apply(lambda x:x.strip())
        return X_copy 

# Logistic Regression

#### Age imputation: random forest

In [5]:
ct = ColumnTransformer(
    [
        ('scaler',StandardScaler(),['new_fare']),
        ('encoding',OneHotEncoder(drop='first'),['pclass','sex_numeric','family_size','age_group','embarked','sex_pclass'])
    ]
)

# Pipeline
pipe_1 = Pipeline(
    [
        ('sex', sextransformer()),
        ('family', familysizetransformer()),
        ('age', agetransformer()),
        ('fare', faretransformer()),
        ('sex_pclass', sexpclasstransformer()),
        ('ct', ct),
        ('lr', LogisticRegression())
    ]
)

# GridsearchCV
param_grid_1 = {
    'lr__C':[5,6,7,8,9],
    'lr__penalty':['l1','l2'],
    'lr__solver':['liblinear','lbfgs'], #兼容性
}

sss = StratifiedShuffleSplit(test_size=0.2, random_state=3)
grid_search_1 = GridSearchCV(estimator=pipe_1, param_grid=param_grid_1, cv=sss, scoring='accuracy')

# fit and find best parameter
X_train = df[['pclass','name','sex','age','sibsp','parch','ticket','fare','embarked']]
y_train = df['survived']
pipe_1.fit(X_train, y_train)
grid_search_1.fit(X_train, y_train)

c:\Users\56864\Desktop\Titanic-2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\56864\Desktop\Titanic-2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\56864\Desktop\Titanic-2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'lr__C': [5, 6, ...], 'lr__penalty': ['l1', 'l2'], 'lr__solver': ['liblinear', 'lbfgs']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedShu...ain_size=None)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parame

In [6]:
# best parameters(sex_pclass:c=8, sex_port:c=5)
print(grid_search_1.best_params_) 
# best model
best_logistic_model = grid_search_1.best_estimator_['lr']
coef = best_logistic_model.coef_
print(coef)
feature_names = grid_search_1.best_estimator_['ct'].get_feature_names_out()
print(feature_names)
df_features = pd.DataFrame(feature_names,coef[0])
print(df_features)
# accuracy (average cv)
print(grid_search_1.best_score_) 
# check overfit
print(grid_search_1.score(X_train,y_train)) 
train_accuracy_1 = grid_search_1.score(X_train,y_train)

{'lr__C': 8, 'lr__penalty': 'l1', 'lr__solver': 'liblinear'}
[[ 0.21389745 -0.44069346 -1.51034655  3.51528042 -1.74501826 -0.00780105
   2.05302612 -0.77515964  0.31723822  0.09267938 -0.44017533  0.
  -1.73574259 -0.42781953 -1.02904987  0.        ]]
['scaler__new_fare' 'encoding__pclass_2' 'encoding__pclass_3'
 'encoding__sex_numeric_1' 'encoding__family_size_large'
 'encoding__family_size_small' 'encoding__age_group_Children'
 'encoding__age_group_Senior' 'encoding__age_group_Teenager'
 'encoding__embarked_Q' 'encoding__embarked_S'
 'encoding__sex_pclass_female2' 'encoding__sex_pclass_female3'
 'encoding__sex_pclass_male1' 'encoding__sex_pclass_male2'
 'encoding__sex_pclass_male3']
                                      0
 0.213897              scaler__new_fare
-0.440693            encoding__pclass_2
-1.510347            encoding__pclass_3
 3.515280       encoding__sex_numeric_1
-1.745018   encoding__family_size_large
-0.007801   encoding__family_size_small
 2.053026  encoding__age_

#### Age imputation: KNN

In [7]:
pipe_2 = Pipeline(
    [
        ('sex', sextransformer()),
        ('family', familysizetransformer()),
        ('age_kkn', agetransformer_kkn()),
        ('fare', faretransformer()),
        ('sex_pclass', sexpclasstransformer()),
        ('ct', ct),
        ('lr', LogisticRegression())
    ]
)

param_grid_2 = {
    'lr__C':[4,5,6,7],
    'lr__penalty':['l1','l2'],
    'lr__solver':['liblinear','lbfgs'], 
}

grid_search_2 = GridSearchCV(estimator=pipe_2, param_grid=param_grid_2, cv=sss, scoring='accuracy')
pipe_2.fit(X_train, y_train)
grid_search_2.fit(X_train, y_train)

c:\Users\56864\Desktop\Titanic-2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\56864\Desktop\Titanic-2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\56864\Desktop\Titanic-2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'lr__C': [4, 5, ...], 'lr__penalty': ['l1', 'l2'], 'lr__solver': ['liblinear', 'lbfgs']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedShu...ain_size=None)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parame

In [8]:
# best parameters (sex_pclass:c=5, sex_port:c=6)
print(grid_search_2.best_params_) 
# best model
best_logistic_model = grid_search_2.best_estimator_['lr']
coef = best_logistic_model.coef_
print(coef)
feature_names = grid_search_2.best_estimator_['ct'].get_feature_names_out()
print(feature_names)
df_features = pd.DataFrame(feature_names,coef[0])
print(df_features)
# accuracy (average cv)
print(grid_search_2.best_score_) 
# check overfit
print(grid_search_2.score(X_train,y_train))
train_accuracy_2 = grid_search_2.score(X_train,y_train) 

{'lr__C': 5, 'lr__penalty': 'l1', 'lr__solver': 'liblinear'}
[[ 0.2286687  -0.39350431 -1.27727211  3.6738298  -1.68422534 -0.01768099
   2.17148997 -0.75563811  0.0955795   0.0631304  -0.44385804  0.
  -1.8325406  -0.22430812 -0.85699656  0.        ]]
['scaler__new_fare' 'encoding__pclass_2' 'encoding__pclass_3'
 'encoding__sex_numeric_1' 'encoding__family_size_large'
 'encoding__family_size_small' 'encoding__age_group_Children'
 'encoding__age_group_Senior' 'encoding__age_group_Teenager'
 'encoding__embarked_Q' 'encoding__embarked_S'
 'encoding__sex_pclass_female2' 'encoding__sex_pclass_female3'
 'encoding__sex_pclass_male1' 'encoding__sex_pclass_male2'
 'encoding__sex_pclass_male3']
                                      0
 0.228669              scaler__new_fare
-0.393504            encoding__pclass_2
-1.277272            encoding__pclass_3
 3.673830       encoding__sex_numeric_1
-1.684225   encoding__family_size_large
-0.017681   encoding__family_size_small
 2.171490  encoding__age_

In [9]:
# test set
df_test = pd.read_csv('test.csv')
df_test.columns = df_test.columns.str.lower()
df_test.info()
df_test.fare = df_test.fare.fillna(df.fare.median())
X_test = df_test[['pclass','name','sex','age','sibsp','parch','ticket','fare','embarked']]
y_predict_1 = grid_search_1.predict(X_test)
predict_result = pd.DataFrame({
    'passengerid':df_test['passengerid'],
    'survived':y_predict_1
})
predict_result.to_csv('predict_result_1.csv',index=False)

y_predict_2 = grid_search_2.predict(X_test)
predict_result = pd.DataFrame({
    'passengerid':df_test['passengerid'],
    'survived':y_predict_2
})
predict_result.to_csv('predict_result_2.csv',index=False)

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerid  418 non-null    int64  
 1   pclass       418 non-null    int64  
 2   name         418 non-null    str    
 3   sex          418 non-null    str    
 4   age          332 non-null    float64
 5   sibsp        418 non-null    int64  
 6   parch        418 non-null    int64  
 7   ticket       418 non-null    str    
 8   fare         417 non-null    float64
 9   cabin        91 non-null     str    
 10  embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB


In [10]:
# streamlit

import joblib

# wrap gridsearch into dictionary
results = {
    'Random Forest':{
        'grid':grid_search_1,
        'test':train_accuracy_1,
        'description':'Impute missing age values using Random Forest'
    },
    'KNN':{
        'grid':grid_search_2,
        'test':train_accuracy_2,
        'description':'Impute missing age values using KNN'
    }
}

joblib.dump(results,'model_analysis_results.pkl')

['model_analysis_results.pkl']